# BÀI THỰC HÀNH 07: EDGE-BASED SEGMENTATION & ACTIVE CONTOUR

1.Mục tiêu bài học
Sau bài thực hành này, sinh viên cần đạt được các mục tiêu sau:
•Hiểu sự khác nhau giữa edge detection vàedge-based segmentation .
•Xây dựng được pipeline hoàn chỉnh đi từ ảnh xám đến edge map , từedge map đếnclosed
contour , và từ contour đến segmentation mask .
•Phân tích được vai trò của các bước tiền xử lý, liên kết biên, lọc contour và điền vùng.
•Hiểu mô hình active contour (snake), bao gồm internal energy ,external energy ,curvature và
ảnh hưởng của initial contour .
•Áp dụng được các kỹ thuật đã học trước đó như histogram, thresholding, filtering, edge
detection, morphology, connected components vào bài toán segmentation dựa trên biên.
2.Kiến thức nền tảng và kết nối với các bài trước
Trong các bài trước, sinh viên đã làm quen với:
•ảnh đa cấp xám và cách lưu ảnh như một ma trận số,
•histogram và ý nghĩa của phân bố mức xám,
•các thao tác trên điểm ảnh và lấy ngưỡng tự động,
•các phép toán số học và logic trên ảnh,
•các bộ lọc làm trơn và làm sắc,
•các thuật toán dò biên như Sobel, Laplacian, Canny.
Bài lab này tiếp nối trực tiếp các nội dung đó. Điểm khác biệt quan trọng là: các thuật toán
dò biên chỉ cho biết biên ở đâu , trong khi segmentation yêu cầu xác định vùng nào là đối tượng .
Vì vậy, phần edge-based segmentation chính là chuỗi bước biến thông tin biên thành biểu diễn
vùng.
2.1. Pipeline tổng quát của lab
Ảnh xám Tiền xử lý
Edge map
Liên kết biên
Contour / Snake
Mask vùng
3
Lab 06B. Edge-Based Segmentation Computer Vision

3.Các phương thức dùng chung cho toàn bộ bài lab
Phần này cung cấp các hàm dùng chung để đọc ảnh, chuẩn hóa dữ liệu và hiển thị trực quan các
kết quả trung gian. Các hàm này nên được đặt ở đầu notebook hoặc đầu file thực hành để tái sử
dụng cho tất cả các phần sau.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.segmentation import active_contour
from skimage.filters import gaussian
import warnings
warnings.filterwarnings('ignore')

def read_gray_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None: raise FileNotFoundError(f"Khong doc duoc anh: {path}")
    return img

def ensure_uint8(img):
    img = np.asarray(img)
    if img.dtype == np.uint8: return img
    return np.clip(img, 0, 255).astype(np.uint8)

def normalize_to_uint8(img):
    img = np.asarray(img).astype(np.float32)
    mn, mx = img.min(), img.max()
    if mx - mn < 1e-8: return np.zeros_like(img, dtype=np.uint8)
    return ((img - mn) / (mx - mn) * 255.0).astype(np.uint8)

def show_images(images, titles=None, cols=3, figsize=(15, 8), cmap="gray"):
    n = len(images)
    rows = int(np.ceil(n / cols))
    plt.figure(figsize=figsize)
    for i, img in enumerate(images):
        plt.subplot(rows, cols, i + 1)
        if img.ndim == 3: plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else: plt.imshow(img, cmap=cmap)
        if titles: plt.title(titles[i])
        plt.axis("off")
    plt.tight_layout()
    plt.show()

def draw_contours_on_image(img_gray, contours, color=(0, 255, 0), thickness=2):
    base = cv2.cvtColor(ensure_uint8(img_gray), cv2.COLOR_GRAY2BGR)
    cv2.drawContours(base, contours, -1, color, thickness)
    return base

def overlay_mask_on_image(img_gray, mask, alpha=0.4):
    base = cv2.cvtColor(ensure_uint8(img_gray), cv2.COLOR_GRAY2BGR)
    overlay = base.copy()
    overlay[mask > 0] = (255, 0, 0)
    return cv2.addWeighted(overlay, alpha, base, 1 - alpha, 0)

def contours_to_mask(img_shape, contours):
    mask = np.zeros(img_shape, dtype=np.uint8)
    cv2.drawContours(mask, contours, -1, 255, thickness=-1)
    return mask

def snake_to_mask(img_shape, snake):
    mask = np.zeros(img_shape, dtype=np.uint8)
    pts = np.round(snake[:, ::-1]).astype(np.int32).reshape((-1, 1, 2))
    cv2.fillPoly(mask, [pts], 255)
    return mask


4.Contour Maps and Edge Representations
4.1. Gợi ý lý thuyết
Mục tiêu của phần này là giúp sinh viên nhận ra rằng bản đồ biên (edge map) chỉ là biểu diễn ban
đầu của đường biên, chứ chưa phải kết quả segmentation. Một edge detector thường tạo ra một
trong ba dạng biểu diễn sau:
•Gradient magnitude map : lưu độ mạnh của biên tại từng pixel.
•Binary edge map : ảnh nhị phân, pixel biên bằng 255 và nền bằng 0.
•Contour representation : biểu diễn dạng danh sách điểm theo thứ tự dọc theo đường biên.
6
Lab 06B. Edge-Based Segmentation Computer Vision
Nếu gọi ảnh xám là I(x, y), đạo hàm theo hai hướng được ký hiệu là GxvàGy, thì độ lớn
gradient là:
|∇I|=√
G2
x+G2
y.
Từ gradient magnitude, ta có thể lấy ngưỡng để thu được ảnh biên nhị phân. Trong các hệ
thống thực tế, Canny thường cho kết quả biên mảnh và ổn định hơn so với lấy ngưỡng đơn giản
trên Sobel magnitude.
4.2. Pipeline sử dụng trong lab
Ảnh xám
Làm trơn
Gradient map
Edge map
Contour extraction

In [ ]:
def compute_gradient_magnitude(img_gray):
    gx = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=3)
    return normalize_to_uint8(np.sqrt(gx**2 + gy**2))

def gradient_threshold_edge_map(grad, thresh=50):
    edge = np.zeros_like(grad, dtype=np.uint8)
    edge[grad >= thresh] = 255
    return edge

def canny_edge_map(img_gray, low=80, high=160, blur_kernel=5):
    blur = cv2.GaussianBlur(img_gray, (blur_kernel, blur_kernel), 1.0)
    return blur, cv2.Canny(blur, low, high)
    
def extract_external_contours(edge_map):
    contours, hierarchy = cv2.findContours(edge_map, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return contours, hierarchy


4.4. Gợi ý loại ảnh mẫu
Nên chọn các loại ảnh có biên rõ, nền đơn giản, ít vật thể chồng lấn:
•đồng xu trên nền tối hoặc sáng tương đối đồng nhất,
•bu lông, con ốc, dụng cụ cơ khí nhỏ,
•lá cây trên nền trắng,
•các vật thể công nghiệp tách rời nhau.
4.5. Bài tập mở rộng
Bài 1: So sánh các biểu diễn biên
•Đầu vào: một ảnh coin hoặc ảnh object có biên rõ.
•Yêu cầu: tạo ba kết quả: gradient magnitude, edge map từ threshold gradient, và edge map
từ Canny.
•Đầu ra: ba ảnh kết quả và một đoạn nhận xét ngắn.
•Mục tiêu: giúp sinh viên thấy rằng gradient magnitude chứa thông tin mức độ mạnh của biên,
còn edge map nhị phân chỉ giữ quyết định có/không có biên.
•Cách so sánh: so sánh độ mảnh của biên, độ liên tục của biên, lượng nhiễu và khả năng phân
biệt biên thật với nền.
8
Lab 06B. Edge-Based Segmentation Computer Vision
Bài 2: Vẽ contour lên ảnh gốc
•Đầu vào: ảnh object đơn giản với nền ít nhiễu.
•Yêu cầu: dùng findContours để trích contour từ edge map và vẽ contour đó lên ảnh gốc.
•Đầu ra: ảnh gốc có overlay contour.
•Mục tiêu: giúp sinh viên hiểu contour là biểu diễn hình học có thứ tự của các điểm biên.
•Ảnh nên chọn: coin, lá cây, hạt hoặc vật hình tròn, hình elip.

### Bài 1: So sánh các biểu diễn biên

In [ ]:
img_4_1 = read_gray_image('../Resources/coins1.jpg')
grad_4_1 = compute_gradient_magnitude(img_4_1)
edge_4_1 = gradient_threshold_edge_map(grad_4_1, thresh=100)
_, canny_4_1 = canny_edge_map(img_4_1, 50, 150)
show_images([img_4_1, grad_4_1, edge_4_1, canny_4_1],
            ['Anh goc', 'Gradient Magnitude', 'Threshold Edge', 'Canny Edge Map'], cols=4, figsize=(20, 5))
print("Nhận xét: Gradient Magnitude thể hiện độ mạnh yếu khác nhau của viền (mức xám đa dạng). Edge Map khi dùng ngưỡng đã loại được một số nét mờ, nhưng nét viền lại rất dày. Canny edge trả về nét mảnh chuẩn xác 1 px.")

### Bài 2: Vẽ contour lên ảnh gốc

In [ ]:
contours_4_2, _ = extract_external_contours(canny_4_1)
ol_4_2 = draw_contours_on_image(img_4_1, contours_4_2, color=(255,0,0))
show_images([img_4_1, ol_4_2], ['Anh goc', 'Contour Overlay'], cols=2, figsize=(10, 5))

5.Edge Linking and Gap Closing
5.1. Gợi ý lý thuyết
Trong thực tế, edge map hiếm khi hoàn hảo. Biên có thể bị đứt do nhiễu, tương phản yếu, ngưỡng
chưa phù hợp hoặc do bề mặt vật thể phản xạ không đều. Nếu các đoạn biên không được nối lại,
contour trích xuất được có thể hở và không dùng tốt cho segmentation.
Gap closing là bước nối các đoạn biên gần nhau. Trong lab này, ta sử dụng cách đơn giản
nhưng hiệu quả là morphological closing , tức là dilation rồi erosion. Bước này có thể làm các
khoảng hở nhỏ được lấp lại. Sau đó, ta dùng connected component filtering để loại bỏ các mảnh
biên nhỏ do nhiễu.
5.2. Pipeline sử dụng trong lab
Edge map
Loại nhiễu nhỏ
Morphological closing
Connected components
Linked edges

In [ ]:
def edge_linking_with_closing(edge_map, kernel_size=5):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    return cv2.morphologyEx(edge_map, cv2.MORPH_CLOSE, kernel)

def filter_small_edge_components(binary_img, min_area=30):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_img, connectivity=8)
    clean = np.zeros_like(binary_img)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] >= min_area: clean[labels == i] = 255
    return clean


5.4. Gợi ý loại ảnh mẫu
Nên chọn ảnh có biên gần khép kín nhưng bị hở nhẹ, ví dụ:
•ảnh coin có phản xạ không đều,
•ảnh hạt, viên thuốc, viên bi,
•ảnh lá cây có mép không quá phức tạp.
Để minh họa lỗi của gap closing quá mạnh, nên có thêm ảnh chứa nhiều object gần nhau.
5.5. Bài tập mở rộng
Bài 1: So sánh trước và sau edge linking
•Đầu vào: một ảnh coin hoặc vật thể hình tròn có biên bị hở nhẹ.
•Yêu cầu: hiển thị edge map trước linking và linked edge map sau closing.
•Đầu ra: hai ảnh và một đoạn nhận xét.
•Mục tiêu: thấy rõ vì sao edge map tốt cho segmentation cần có biên đủ liên tục.
•Cách so sánh: số lượng khoảng hở, độ liên tục của contour, mức nhiễu.
Bài 2: Ảnh hưởng của kích thước kernel
•Đầu vào: ảnh có nhiều vật thể gần nhau.
•Yêu cầu: thử3×3,5×5,7×7.
•Đầu ra: ba ảnh linked edge map và ba kết quả contour tương ứng.
10
Lab 06B. Edge-Based Segmentation Computer Vision
•Mục tiêu: phân tích sự đánh đổi giữa việc lấp khoảng hở và nguy cơ làm dính hai object khác
nhau.
•Ảnh nên chọn: coin gần nhau, viên thuốc, hạt nhỏ.

### Bài 1: So sánh trước và sau edge linking

In [ ]:
img_5_1 = read_gray_image('../Resources/leaves (1).png')
_, canny_5_1 = canny_edge_map(img_5_1, 50, 100, blur_kernel=3)
linked_5_1 = edge_linking_with_closing(canny_5_1, 5)

show_images([canny_5_1, linked_5_1], ["Edge map trưóc linking (Bị hở đứt)", "Linked Edge map sau closing (Hàn liền)"], cols=2, figsize=(12, 6))
print("Nhận xét: Morphological closing giúp các phân mảnh hở nối liền.")

### Bài 2: Ảnh hưởng của kích thước kernel

In [ ]:
img_5_2 = read_gray_image('../Resources/coins2.jpg')
_, canny_5_2 = canny_edge_map(img_5_2, 80, 150)

linked_3 = edge_linking_with_closing(canny_5_2, kernel_size=3)
linked_5 = edge_linking_with_closing(canny_5_2, kernel_size=5)
linked_7 = edge_linking_with_closing(canny_5_2, kernel_size=7)

cts_3, _ = extract_external_contours(linked_3)
cts_5, _ = extract_external_contours(linked_5)
cts_7, _ = extract_external_contours(linked_7)

show_images([linked_3, linked_5, linked_7, 
             draw_contours_on_image(img_5_2, cts_3), draw_contours_on_image(img_5_2, cts_5), draw_contours_on_image(img_5_2, cts_7)], 
            ["Linked Map (3x3)", "Linked (5x5)", "Linked (7x7)", 
             "Contour Overlay (3x3)", "Contour Overlay (5x5)", "Contour Overlay (7x7)"], cols=3, figsize=(15, 10))

6.From Edge Maps to Closed Boundaries
6.1. Gợi ý lý thuyết
Phần này kết nối trực tiếp giữa edge detection và contour-based segmentation. Một edge map chỉ
là tập các pixel biên; còn segmentation yêu cầu một đường bao đủ tốt để phân chia miền trong
và miền ngoài. Vì vậy, mục tiêu ở đây là biến edge map thành một tập closed boundaries hoặc ít
nhất là contour đủ ổn định để tiếp tục xử lý.
Các bước chính gồm: làm trơn ảnh, phát hiện biên, làm sạch edge map, nối các khe hở nhỏ,
trích contour và chọn các contour hợp lệ.
6.2. Pipeline sử dụng trong lab
Image
Smoothing
Edge detection
Cleaning
Gap closing
Contours

In [ ]:
def edge_to_closed_boundary_pipeline(img_gray, canny1=80, canny2=160, min_comp_area=30, min_contour_area=100, kernel_size=5):
    blur = cv2.GaussianBlur(img_gray, (5, 5), 1.0)
    edges = cv2.Canny(blur, canny1, canny2)
    linked = edge_linking_with_closing(edges, kernel_size=kernel_size)
    clean = filter_small_edge_components(linked, min_area=min_comp_area)
    contours, _ = cv2.findContours(clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    good_contours = [c for c in contours if cv2.contourArea(c) >= min_contour_area]
    return {"edges": edges, "linked": linked, "clean_edges": clean, "contours": good_contours, "overlay": draw_contours_on_image(img_gray, good_contours)}


6.4. Gợi ý loại ảnh mẫu
Chọn các ảnh có object với đường biên gần khép kín, ví dụ coin, screw, washer, pill. Nếu muốn
minh họa trường hợp khó hơn, có thể thêm ảnh vật thể có vài đoạn biên yếu.
6.5. Bài tập mở rộng
Bài 1: Pipeline có và không có gap closing
•Đầu vào: ảnh coin, screw hoặc seed.
•Yêu cầu: chạy pipeline với và không với bước morphological closing.
•Đầu ra: contour overlay của hai trường hợp.
•Mục tiêu: chứng minh bước gap closing là cần thiết để tạo closed boundaries.
•Cách so sánh: số contour hợp lệ, độ kín của contour, khả năng dùng contour để fill vùng.
Bài 2: Lọc contour hợp lệ
•Đầu vào: ảnh có cả biên object thật và biên nhiễu.
•Yêu cầu: thay đổi ngưỡng min_contour_area .
•Đầu ra: các contour còn lại sau lọc.
•Mục tiêu: giúp sinh viên hiểu vai trò của bước lựa chọn contour trong pipeline segmentation.

### Bài 1: Pipeline có và không có gap closing

In [ ]:
img_6_1 = read_gray_image('../Resources/seed1.png')
out_no_gap = edge_to_closed_boundary_pipeline(img_6_1, 50, 100, kernel_size=1) 
out_with_gap = edge_to_closed_boundary_pipeline(img_6_1, 50, 100, kernel_size=9)

show_images([out_no_gap['overlay'], out_with_gap['overlay']], ["Contour Overlay (Không Gap Closing)", "Contour Overlay (Có Gap Closing K=9)"], cols=2, figsize=(15, 6))

### Bài 2: Lọc contour hợp lệ

In [ ]:
out_area_10 = edge_to_closed_boundary_pipeline(img_6_1, 50, 100, min_contour_area=10)
out_area_500 = edge_to_closed_boundary_pipeline(img_6_1, 50, 100, min_contour_area=500)

show_images([out_area_10['overlay'], out_area_500['overlay']], ["Contour còn lại lọc area=10", "Contour còn lại lọc area=500"], cols=2, figsize=(15, 6))

7.Closed Contours and Region Filling
7.1. Gợi ý lý thuyết
Khi đã có contour khép kín hoặc đủ kín để coi như khép kín, ta có thể tô vùng bên trong contour
để tạo segmentation mask. Đây là bước chuyển từ biểu diễn theo đường biên sang biểu diễn theo
vùng . Bài này giúp sinh viên nhìn thấy rõ rằng một đường bao có ý nghĩa hình học và có thể sử
dụng trực tiếp để xác định vùng của đối tượng.
Trong nhiều bài toán cơ bản, closed contour segmentation hoạt động tốt nếu:
12
Lab 06B. Edge-Based Segmentation Computer Vision
•object có biên rõ,
•object không chồng lấn mạnh,
•contour sau linking đủ kín.
7.2. Pipeline sử dụng trong lab
Closed contours
Contour filtering
Region filling
Segmentation mask

In [ ]:
def contour_circularity(contour):
    area = cv2.contourArea(contour)
    perim = cv2.arcLength(contour, True)
    if perim < 1e-8: return 0.0
    return 4.0 * np.pi * area / (perim ** 2)

def filter_circular_contours(contours, min_area=100, min_circularity=0.75):
    return [c for c in contours if cv2.contourArea(c) >= min_area and contour_circularity(c) >= min_circularity]


7.4. Gợi ý loại ảnh mẫu
Ảnh phù hợp nhất cho phần này là các object tách rời, biên đơn giản, ví dụ:
•đồng xu,
•viên thuốc,
•hạt giống,
•chi tiết cơ khí nhỏ,
•shape hình học in trên giấy.
7.5. Bài tập mở rộng
Bài 1: Lọc contour theo diện tích và chu vi
•Đầu vào: ảnh chứa nhiều vật thể và một số contour nhiễu.
•Yêu cầu: viết hàm lọc contour theo diện tích tối thiểu và chu vi tối thiểu.
•Đầu ra: mask chỉ chứa các object hợp lệ.
•Mục tiêu: giúp sinh viên thực hành phân tích hình học của contour và loại bỏ biên không
mong muốn.
•So sánh: số contour ban đầu và số contour sau lọc, chất lượng mask sau khi fill.
Bài 2: Chỉ giữ vật thể gần tròn
•Đầu vào: ảnh chứa cả object tròn và không tròn.
•Yêu cầu: dùng circularity để chỉ giữ vật thể có dạng gần tròn.
•Đầu ra: mask và ảnh overlay trên ảnh gốc.
•Mục tiêu: kết hợp contour extraction với shape analysis.
•Ảnh nên chọn: ảnh coin trộn với bu lông, đai ốc, hoặc ảnh nhiều shape in trên giấy.
14
Lab 06B. Edge-Based Segmentation Computer Vision

### Bài 1: Lọc contour theo diện tích và chu vi

In [ ]:
img_7_1 = read_gray_image('../Resources/leaves2.png')
out_7_1 = edge_to_closed_boundary_pipeline(img_7_1, kernel_size=5, min_contour_area=0)
all_contours = out_7_1['contours']

def filter_area_perim(contours, min_area, min_perim):
    return [c for c in contours if cv2.contourArea(c) >= min_area and cv2.arcLength(c, True) >= min_perim]

good_ct = filter_area_perim(all_contours, min_area=600, min_perim=150)
mask_7_1 = contours_to_mask(img_7_1.shape, good_ct)

print(f"Số lượng contour ban đầu: {len(all_contours)}")
print(f"Số lượng contour sau khi lọc theo DT+ChuVi: {len(good_ct)}")
show_images([draw_contours_on_image(img_7_1, all_contours), draw_contours_on_image(img_7_1, good_ct), mask_7_1], 
            ['Tat ca contour tren anh goc', 'Cac contour sau loc', 'Mask chứa Object hop le'], cols=3, figsize=(15, 5))

### Bài 2: Chỉ giữ vật thể gần tròn

In [ ]:
img_7_2 = read_gray_image('../Resources/coins3.png')
out_7_2 = edge_to_closed_boundary_pipeline(img_7_2, 50, 100, kernel_size=7)

circular_ct = filter_circular_contours(out_7_2['contours'], min_area=300, min_circularity=0.6)
mask_7_2 = contours_to_mask(img_7_2.shape, circular_ct)

show_images([img_7_2, mask_7_2, overlay_mask_on_image(img_7_2, mask_7_2)], 
            ['Ảnh Gốc (Mix Đồ tròn/dẹt)', 'Mask (Chỉ Vật Thể Tròn)', 'Ảnh Overlay Trên Ảnh Gốc'], cols=3, figsize=(15, 5))

8.Active Contour (Snake) Segmentation
8.1. Gợi ý lý thuyết
Khi biên của object không đủ mạnh hoặc contour không dễ đóng kín chỉ bằng edge detection và
gap closing, ta cần một mô hình linh hoạt hơn: active contour haysnake . Snake là một đường
cong động có thể co, giãn và biến dạng dần để tiến tới ranh giới của object.
Snake tối ưu tổng năng lượng:
Esnake =Einternal +Eexternal .
8.1.1. Internal energy
Internal energy giữ cho contour có tính hình học hợp lý. Một dạng viết thường gặp là:
Einternal =α|v′(s)|2+β|v′′(s)|2.
Trong đó:
•v′(s)liên quan đến độ căng của contour;
•v′′(s)liên quan đến độ cong và độ mượt.
Ý nghĩa trực giác:
•αlớn làm contour có xu hướng ngắn lại, tránh kéo dài bất thường;
•βlớn làm contour mượt hơn, ít bị gấp khúc.
8.1.2. External energy
External energy được lấy từ ảnh, thường dựa trên gradient hoặc một force map . Một biểu thức
phổ biến là:
Eexternal =−|∇I(x, y)|2.
Nghĩa là snake có xu hướng di chuyển về nơi gradient lớn, tức là các vị trí có biên mạnh.
8.1.3. Curvature và vai trò của độ mượt
Thành phần curvature liên quan đến đạo hàm bậc hai của contour. Nó giúp contour tránh bị răng
cưa hoặc bám theo nhiễu cục bộ. Nếu regularization quá yếu, snake có thể bị nhiễu kéo lệch; nếu
quá mạnh, snake trở nên cứng và khó bám sát biên thật.
8.1.4. Initial contour
Snake luôn cần một contour ban đầu. Nếu contour khởi tạo gần object, kết quả thường tốt. Nếu
khởi tạo quá xa hoặc cắt ngang object, snake có thể hội tụ sai. Đây là điểm cần được làm rõ bằng
ví dụ thực nghiệm trong lab.
15
Lab 06B. Edge-Based Segmentation Computer Vision
8.2. Pipeline sử dụng trong lab
Ảnh xám
Làm trơn
Image force map
Initial contour
Iterative deformation
Final snake

In [ ]:
def run_active_contour(img_gray, init_contour, sigma=2.0, alpha=0.1, beta=0.2, gamma=0.01):
    img_float = img_gray.astype(np.float32) / 255.0
    img_smooth = gaussian(img_float, sigma=sigma)
    snake = active_contour(img_smooth, init_contour, alpha=alpha, beta=beta, gamma=gamma, coordinates='rc')
    return img_smooth, snake

def create_ellipse_init_contour(center_row, center_col, radius_row, radius_col, num_points=200):
    t = np.linspace(0, 2 * np.pi, num_points)
    return np.stack([center_row + radius_row * np.sin(t), center_col + radius_col * np.cos(t)], axis=1)

def draw_snake_on_image(img_gray, snake_points, color=(0, 255, 0), thickness=2):
    base = cv2.cvtColor(ensure_uint8(img_gray), cv2.COLOR_GRAY2BGR)
    pts = np.round(snake_points[:, ::-1]).astype(np.int32).reshape((-1, 1, 2))
    cv2.polylines(base, [pts], isClosed=True, color=color, thickness=thickness)
    return base


8.4. Gợi ý loại ảnh mẫu
Snake phù hợp với những ảnh mà biên tương đối liên tục nhưng không quá sắc, hoặc khó khép
kín chỉ bằng contour filling thông thường, chẳng hạn:
•ảnh cell,
•ảnh cơ quan hoặc cấu trúc giải phẫu đơn giản,
•ảnh vật thể tròn hoặc gần tròn có biên mềm,
•ảnh object đơn lẻ trên nền ít nhiễu.
8.5. Bài tập mở rộng
Bài 1: So sánh initial contour tốt và xấu
•Đầu vào: ảnh chỉ có một object chính, ví dụ cell hoặc đồng xu.
•Yêu cầu: tạo hai initial contour: một contour gần object và một contour xa object.
17
Lab 06B. Edge-Based Segmentation Computer Vision
•Đầu ra: hai snake overlay và hai mask tương ứng.
•Mục tiêu: chứng minh rằng khởi tạo ban đầu ảnh hưởng mạnh tới khả năng hội tụ của snake.
•Cách so sánh: vị trí contour cuối cùng, mức độ bám biên, mức độ sai lệch khỏi object thật.
Bài 2: Ảnh hưởng của α,β,γ
•Đầu vào: một ảnh và một initial contour cố định.
•Yêu cầu: thay đổi từng tham số trong khi giữ các tham số còn lại cố định.
•Đầu ra: các snake overlay tương ứng với từng bộ tham số.
•Mục tiêu: hiểu vai trò của tension, smoothness và tốc độ cập nhật trong mô hình snake.
•Ảnh nên chọn: cell đơn lẻ, coin, hoặc vật gần tròn.
Bài 3: So sánh contour filling và snake
•Đầu vào: ảnh có biên tương đối yếu.
•Yêu cầu: áp dụng cả closed contour filling và active contour.
•Đầu ra: hai mask kết quả và ảnh overlay tương ứng.
•Mục tiêu: giúp sinh viên hiểu khi nào snake phù hợp hơn contour-based segmentation đơn
giản.
•Cách so sánh: độ bám theo biên thật, tính ổn định khi biên bị hở hoặc yếu.

### Bài 1: So sánh initial contour tốt và xấu

In [ ]:
img_8_1 = cv2.resize(read_gray_image('../Resources/cell1.jpg'), (256, 256))
init_good = create_ellipse_init_contour(130, 130, 80, 80)
init_bad = create_ellipse_init_contour(50, 50, 30, 30)

_, snake_good = run_active_contour(img_8_1, init_good, alpha=0.015, beta=10, gamma=0.001)
_, snake_bad = run_active_contour(img_8_1, init_bad, alpha=0.015, beta=10, gamma=0.001)

sg_mask = snake_to_mask(img_8_1.shape, snake_good)
sb_mask = snake_to_mask(img_8_1.shape, snake_bad)

show_images([draw_snake_on_image(img_8_1, snake_good), sg_mask, 
             draw_snake_on_image(img_8_1, snake_bad), sb_mask],
            ['Overlay (Good init)', 'Mask (Good Init)', 'Overlay (Bad Init)', 'Mask (Bad Init)'], cols=2, figsize=(10, 10))

### Bài 2: Ảnh hưởng của alpha, beta, gamma

In [ ]:
_, snake_a1 = run_active_contour(img_8_1, init_good, alpha=0.5, beta=10, gamma=0.001)     # High alpha (Elasticity)
_, snake_b1 = run_active_contour(img_8_1, init_good, alpha=0.015, beta=100, gamma=0.001)  # High beta (Rigidity)
_, snake_g1 = run_active_contour(img_8_1, init_good, alpha=0.015, beta=10, gamma=0.1)     # High gamma (Edge Attraction)

show_images([draw_snake_on_image(img_8_1, snake_a1), draw_snake_on_image(img_8_1, snake_b1), draw_snake_on_image(img_8_1, snake_g1)],
            ['High Alpha (Căng rút lõm)', 'High Beta (Snake gồng mình cứng)', 'High Gamma (Lực hút nhạy biên, răng cưa)'], cols=3, figsize=(15, 5))

### Bài 3: So sánh contour filling và snake trên ảnh biên yếu

In [ ]:
img_8_3 = cv2.resize(read_gray_image('../Resources/WBC2.jpg'), (256, 256))
out_8_3 = edge_to_closed_boundary_pipeline(img_8_3, 30, 80, kernel_size=5)
mask_contour = contours_to_mask(img_8_3.shape, out_8_3['contours'])

snake_init_8_3 = create_ellipse_init_contour(130, 130, 90, 90)
_, snake_c = run_active_contour(img_8_3, snake_init_8_3, alpha=0.015, beta=5, gamma=0.005)
mask_snake = snake_to_mask(img_8_3.shape, snake_c)

show_images([mask_contour, overlay_mask_on_image(img_8_3, mask_contour), mask_snake, overlay_mask_on_image(img_8_3, mask_snake)], 
            ["Mask ket qua (Contour)", "Overlay (Contour)", "Mask ket qua (Snake)", "Overlay (Snake)"], cols=2, figsize=(15, 10))

9.Bài tập tổng hợp cho toàn bộ bài học
Bài 1: Xây dựng pipeline contour-based segmentation hoàn chỉnh
•Đầu vào: ảnh coin hoặc ảnh chi tiết cơ khí có nền đơn giản.
•Yêu cầu: xây dựng pipeline gồm Gaussian blur, Canny, gap closing, contour extraction,
contour filtering và region filling.
•Đầu ra: edge overlay, contour overlay, segmentation mask và mask overlay.
•Mục tiêu: thực hành toàn bộ pipeline chuẩn của edge-based segmentation.
•Cách so sánh: đối chiếu các ảnh trung gian để giải thích bước nào đóng vai trò quan trọng
nhất.
Bài 2: Nghiên cứu ảnh hưởng của gap closing
•Đầu vào: ảnh object có biên hở nhẹ.
•Yêu cầu: chạy cùng một pipeline với các kernel closing khác nhau.
•Đầu ra: bảng hình ảnh gồm edge map, linked edges, contour overlay và mask ở các cấu hình
khác nhau.
•Mục tiêu: phân tích mối liên hệ giữa edge linking và khả năng khép contour.
18
Lab 06B. Edge-Based Segmentation Computer Vision
Bài 3: Shape-aware contour selection
•Đầu vào: ảnh chứa nhiều object có hình dạng khác nhau.
•Yêu cầu: xây dựng tiêu chí giữ lại object gần tròn hoặc object có diện tích trong khoảng xác
định.
•Đầu ra: mask chỉ chứa nhóm object mong muốn.
•Mục tiêu: thực hành phân tích hình học trên contour và ra quyết định theo bài toán cụ thể.
Bài 4: Snake trên ảnh biên yếu
•Đầu vào: ảnh cell hoặc object có biên mềm.
•Yêu cầu: chạy snake với hai initial contour khác nhau và nhận xét.
•Đầu ra: snake overlay, mask snake và phần mô tả sự khác nhau giữa hai trường hợp.
•Mục tiêu: nhấn mạnh vai trò của khởi tạo ban đầu trong active contour.
Bài 5: Case study nhỏ trên một tập ảnh
•Đầu vào: chọn một tập ảnh nhỏ gồm 5–10 ảnh cùng loại, ví dụ coins, leaves hoặc cells.
•Yêu cầu: áp dụng ít nhất hai hướng: closed contour filling và snake.
•Đầu ra: bộ kết quả cho từng ảnh và nhận xét tổng hợp về ưu/nhược điểm của từng hướng.
•Mục tiêu: giúp sinh viên không chỉ chạy code trên một ảnh đơn lẻ mà còn đánh giá tính ổn
định của thuật toán trên một nhóm ảnh.
•Cách so sánh: dùng tiêu chí định tính như độ bám biên, mức nhiễu của mask, khả năng tách
nền và độ nhạy với tham số.

### Bài 1: Pipeline contour-based segmentation hoàn chỉnh

In [ ]:
img_9_1 = read_gray_image('../Resources/coins1.jpg')
out_9_1 = edge_to_closed_boundary_pipeline(img_9_1, 50, 150, min_comp_area=50, min_contour_area=200, kernel_size=9)
mask_9_1 = contours_to_mask(img_9_1.shape, out_9_1['contours'])
overlay_mask_9_1 = overlay_mask_on_image(img_9_1, mask_9_1)
edge_overlay = overlay_edges_on_image(img_9_1, out_9_1['edges'])

show_images([edge_overlay, out_9_1['overlay'], mask_9_1, overlay_mask_9_1],
            ['Edge Overlay', 'Contour Overlay', 'Segmentation Mask', 'Mask Overlay'], cols=4, figsize=(20, 5))

### Bài 2: Nghiên cứu ảnh hưởng của gap closing

In [ ]:
out_k3 = edge_to_closed_boundary_pipeline(img_9_1, 50, 150, kernel_size=3)
out_k9 = edge_to_closed_boundary_pipeline(img_9_1, 50, 150, kernel_size=9)

show_images([out_k3['edges'], out_k3['linked'], out_k3['overlay'], contours_to_mask(img_9_1.shape, out_k3['contours'])], 
            ["Edge Map (K=3)", "Linked Edges (K=3)", "Contour Overlay (K=3)", "Mask (K=3)"], cols=4, figsize=(20, 5))
            
show_images([out_k9['edges'], out_k9['linked'], out_k9['overlay'], contours_to_mask(img_9_1.shape, out_k9['contours'])], 
            ["Edge Map (K=9)", "Linked Edges (K=9) - Better", "Contour Overlay (K=9)", "Mask (K=9)"], cols=4, figsize=(20, 5))

### Bài 3: Shape-aware contour selection

In [ ]:
img_9_3 = read_gray_image('../Resources/coins3.png')
out_9_3 = edge_to_closed_boundary_pipeline(img_9_3, 40, 120, min_contour_area=200, kernel_size=5)

# Selection: Chỉ giữ Object hình Thon dài hoặc không phải Tròn Đồng Xu.
non_circular = [c for c in out_9_3['contours'] if contour_circularity(c) < 0.6 and cv2.contourArea(c) > 500]
final_mask_9_3 = contours_to_mask(img_9_3.shape, non_circular)

show_images([img_9_3, final_mask_9_3], ['Ảnh Gốc Nhỏ + To', 'Mask: Chỉ chứa object Thon dẹt mong muốn'], cols=2, figsize=(15, 6))

### Bài 4: Snake trên ảnh biên yếu

In [ ]:
img_9_4 = cv2.resize(read_gray_image('../Resources/WBC1.jpg'), (256, 256))
init_a = create_ellipse_init_contour(130, 130, 60, 60)
init_b = create_ellipse_init_contour(130, 130, 120, 120)

_, s1 = run_active_contour(img_9_4, init_a, alpha=0.01, beta=0.5, gamma=0.005)
_, s2 = run_active_contour(img_9_4, init_b, alpha=0.01, beta=0.5, gamma=0.005)

mask_s1 = snake_to_mask(img_9_4.shape, s1)
mask_s2 = snake_to_mask(img_9_4.shape, s2)

show_images([draw_snake_on_image(img_9_4, s1), mask_s1, draw_snake_on_image(img_9_4, s2), mask_s2],
            ['Snake S1 Overlay', 'Mask S1', 'Snake S2 Overlay', 'Mask S2'], cols=4, figsize=(20, 5))
print("Sự khác biệt: Nếu ban đầu tạo vòng nhỏ (S1), snake nở ra có thể bắt đúng nhân Tế Bào, nhưng nếu làm quá to (S2) bao hết thì Rắn lại bám vào màng rìa ngoài cùng mờ nhạt do lực hút phân tán.")

### Bài 5: Case study nhỏ trên một tập ảnh (Ảnh Hạt/Lá/Tiền)

In [ ]:
test_paths = ['../Resources/seed1.png', '../Resources/leaves (1).png', '../Resources/coins1.jpg']
for p in test_paths:
    img = read_gray_image(p)
    out = edge_to_closed_boundary_pipeline(img, 40, 120, kernel_size=5)
    
    # Run active contour (Snake)
    img_rs = cv2.resize(img, (256, 256))
    init_snake = create_ellipse_init_contour(128, 128, 80, 80)
    _, snake_final = run_active_contour(img_rs, init_snake, alpha=0.02, beta=10, gamma=0.005)
    
    show_images([img, out['overlay'], contours_to_mask(img.shape, out['contours']), draw_snake_on_image(img_rs, snake_final)], 
                [f'Ảnh: {os.path.basename(p)}', 'Contour Overlay', 'Contour Mask', 'Snake Approach'], cols=4, figsize=(20, 4))
print("Nhận xét: Snake phù hợp ảnh có 1 object to giữa khung, còn Region filling Contour giải quyết gọn đa vật thể (coins).")

10.Kết luận
Qua bài lab này, sinh viên cần rút ra các điểm chính sau:
•Dò biên không đồng nghĩa với segmentation; cần thêm nhiều bước trung gian để biến thông
tin biên thành vùng.
•Biểu diễn contour là cầu nối quan trọng giữa edge map vàregion mask .
•Gap closing và contour filtering là hai bước rất quan trọng trong các pipeline edge-based cổ
điển.
•Active contour là công cụ mạnh khi biên yếu hoặc khi đường bao không dễ thu được chỉ bằng
edge map nhị phân.
•Không có phương pháp nào tốt cho mọi loại ảnh; việc chọn pipeline phụ thuộc mạnh vào đặc
trưng biên của đối tượng, độ phức tạp của nền và mục tiêu phân đoạn.
19